# 03 — Forecast Analysis

This notebook visualizes and compares the Monday forecasts produced in the modeling notebook.

Run `02_exchange_rate_forecasting.ipynb` first to generate the required result files.

## Setup and result loading

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "currency_explorer").exists():
    project_root = project_root.parent

results_dir = project_root / "results"
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    results_dir / "predictions.csv",
    results_dir / "metrics.csv",
    results_dir / "sarima_config.json",
]

missing_files = [
    path.name for path in required_files if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Run notebook 02 first. Missing: "
        + ", ".join(missing_files)
    )

print("Results directory:", results_dir)

In [ ]:
predictions_df = pd.read_csv(
    results_dir / "predictions.csv",
    parse_dates=["forecast_date", "previous_date"],
)
metrics_df = pd.read_csv(
    results_dir / "metrics.csv",
    index_col=0,
)

with open(
    results_dir / "sarima_config.json",
    "r",
    encoding="utf-8",
) as file:
    model_config = json.load(file)

predictions_df = predictions_df.sort_values(
    "forecast_date"
).reset_index(drop=True)

print("Forecasts:", len(predictions_df))
print(
    "Period:",
    predictions_df["forecast_date"].min().date(),
    "—",
    predictions_df["forecast_date"].max().date(),
)
model_config

## Metrics summary

In [ ]:
metrics_df

## Actual rates and model forecasts

Each point represents a Monday for which all three model predictions are available.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    predictions_df["forecast_date"],
    predictions_df["actual_rate"],
    color="black",
    linewidth=2.5,
    label="Actual",
)
ax.plot(
    predictions_df["forecast_date"],
    predictions_df["naive_prediction"],
    linestyle="--",
    label="Naive",
)
ax.plot(
    predictions_df["forecast_date"],
    predictions_df["sarima_prediction"],
    linestyle="--",
    label="SARIMA",
)
ax.plot(
    predictions_df["forecast_date"],
    predictions_df["holt_prediction"],
    linestyle="--",
    label="Holt",
)

ax.set_title("Actual Monday rates and one-step forecasts")
ax.set_xlabel("Forecast date")
ax.set_ylabel("Exchange rate")
ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig(
    figures_dir / "01_forecasts.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## MAE and RMSE comparison

In [ ]:
ax = metrics_df.plot(
    kind="bar",
    figsize=(9, 5),
    rot=0,
)

ax.set_title("Forecast error by model")
ax.set_xlabel("Model")
ax.set_ylabel("Error")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Metric")

fig = ax.get_figure()
fig.tight_layout()
fig.savefig(
    figures_dir / "02_metrics.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## Absolute errors over time

In [ ]:
prediction_columns = {
    "Naive": "naive_prediction",
    "SARIMA": "sarima_prediction",
    "Holt": "holt_prediction",
}

error_df = predictions_df[["forecast_date"]].copy()

for model_name, prediction_column in prediction_columns.items():
    error_df[model_name] = (
        predictions_df["actual_rate"]
        - predictions_df[prediction_column]
    ).abs()

fig, ax = plt.subplots(figsize=(14, 6))

for model_name in prediction_columns:
    ax.plot(
        error_df["forecast_date"],
        error_df[model_name],
        label=model_name,
    )

ax.set_title("Absolute forecast errors over time")
ax.set_xlabel("Forecast date")
ax.set_ylabel("Absolute error")
ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig(
    figures_dir / "03_absolute_errors.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## Error distributions

In [ ]:
ax = error_df[list(prediction_columns)].plot.box(
    figsize=(9, 5),
    showfliers=True,
)

ax.set_title("Distribution of absolute forecast errors")
ax.set_xlabel("Model")
ax.set_ylabel("Absolute error")
ax.grid(axis="y", alpha=0.3)

fig = ax.get_figure()
fig.tight_layout()
fig.savefig(
    figures_dir / "04_error_distributions.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## Final comparison

In [ ]:
best_mae_model = metrics_df["MAE"].idxmin()
best_rmse_model = metrics_df["RMSE"].idxmin()

comparison = pd.DataFrame({
    "criterion": ["Lowest MAE", "Lowest RMSE"],
    "best_model": [best_mae_model, best_rmse_model],
    "value": [
        metrics_df.loc[best_mae_model, "MAE"],
        metrics_df.loc[best_rmse_model, "RMSE"],
    ],
})

comparison

In [ ]:
print("Best model by MAE:", best_mae_model)
print("Best model by RMSE:", best_rmse_model)
print("Figures saved to:", figures_dir)